In [1]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

In [2]:
# Clear any logs from previous runs
!rm -rf ./logs/

'rm' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras
tf.debugging.experimental.enable_dump_debug_info('./logs/',
                                                 tensor_debug_mode="FULL_HEALTH", 
                                                 circular_buffer_size=-1)
from keras import backend as K
import numpy as np

print("TensorFlow version: ", tf.__version__)
assert version.parse(tf.__version__).release[0] >= 2, \
    "This notebook requires TensorFlow 2.0 or above."

INFO:tensorflow:Enabled dumping callback in thread MainThread (dump root: ./logs/, tensor debug mode: FULL_HEALTH)
TensorFlow version:  2.19.1


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

diabetes = load_diabetes()

# Use the BMI feature (index 2)
X = diabetes.data[:, 2:3]
y = diabetes.target

# Standardize the target variable 
y = (y - y.mean()) / y.std()

# Split into test and train pairs
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# The next cell uses train_size for batch_size
train_size = len(x_train)

In [ ]:
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

model = keras.models.Sequential([
    keras.layers.Dense(16, input_dim=1),
    keras.layers.Dense(1),
])

model.compile(
    loss='mse',
    optimizer=keras.optimizers.SGD(learning_rate=0.2),
)

print("Training ... With default parameters, this takes less than 10 seconds.")
training_history = model.fit(
    x_train, 
    y_train,
    batch_size=train_size,
    verbose=1,
    epochs=20,
    validation_data=(x_test, y_test),
    callbacks=[tensorboard_callback],
)

print("Average test loss: ", np.average(training_history.history['loss']))

c:\Users\Hritvik\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training ... With default parameters, this takes less than 10 seconds.
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 1.0544 - val_loss: 0.9280
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - loss: 1.0511 - val_loss: 0.9232
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - loss: 1.0482 - val_loss: 0.9217
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 1.0454 - val_loss: 0.9190
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 1.0426 - val_loss: 0.9168
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 1.0398 - val_loss: 0.9145
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 1.0371 - val_loss: 0.9122
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 1.0343 - val_loss: 0.9100
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - loss: 1.0316 - val_loss: 0.9077
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 1.0290 - val_loss: 0.9055
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - loss: 1.0263 - val_loss: 0.9033
Epoch 12/20

In [7]:
%tensorboard --logdir logs/

A brief overview of the visualizations created in this example and the dashboards (tabs in top navigation bar) where they can be found:

* Scalars show how the loss and metrics change with every epoch. You can use them to also track training speed, learning rate, and other scalar values. Scalars can be found in the Time Series or Scalars dashboards.
* Graphs help you visualize your model. In this case, the Keras graph of layers is shown which can help you ensure it is built correctly. Graphs can be found in the Graphs dashboard.
* Histograms and Distributions show the distribution of a Tensor over time. This can be useful to visualize weights and biases and verify that they are changing in an expected way. Histograms can be found in the Time Series or Histograms dashboards. Distributions can be found in the Distributions dashboard.

Breakdown of the Debugger Interface
The Debugger Dashboard on the Tensorboard consists of five main components:

* __Alerts:__ This top-left section contains a list of alert events detected by the debugger in the debug data from the instrumented TensorFlow program. Each alert indicates a certain anomaly that warrants attention. In our case, this section highlights 499 NaN/âˆž events with a salient pink-red color. This confirms our suspicion that the model fails to learn because of the presence of NaNs and/or infinities in its internal tensor values.
* __Python Execution Timeline:__ This is the upper half of the top-middle section. It presents the full history of the eager execution of ops and graphs. Each box of the timeline is marked by the initial letter of the op or graphâ€™s name. We can navigate this timeline by using the navigation buttons and the scrollbar above the timeline.
* __Graph Execution:__ Located at the top-right corner of the GUI, this section will be central to our debugging task. It contains a history of all the floating-type tensors computed inside graphs, i.e., the ones compiled by @tf-functions.
* __Stack Trace:__ The bottom-right section, shows the stack trace of the creation of every single operation on the graph.
* __Source Code:__ The bottom-left section, highlights the source code corresponding to each operation on the graph.
